# Module 4: Face Detection, Face Verification, and Age-Related Context Handling

This notebook demonstrates and tests **Module 4** of the AI-Based Fake Identity & Document Screening System.

## Module 4 Objectives:
1. Perform pairwise face detection and verification between passport portrait (Image A) and current face photo (Image B).
2. Require exactly one usable face per image.
3. Perform database record lookup (SQLite) using extracted passport number (independently of face verification).
4. Resolve issue date using priority policy (DB issue date → OCR issue date → Unavailable).
5. Calculate document age (elapsed time) relative to reference date.
6. Evaluate 6-case age-context policy and generate manual review recommendations.

---

## Environment & Dependency Check
Run the cell below to load project modules and check if `deepface` is installed.

In [1]:
import sys
from datetime import date
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.face_verification_service import verify_faces_with_context
from app.face_verification.db_lookup import lookup_passport_record
from app.face_verification.date_resolver import resolve_issue_date
from app.face_verification.verifier import verify_faces
from app.face_verification.age_policy import apply_age_context_policy, AgePolicyConfig

print("✅ Project Root:", PROJECT_ROOT)
try:
    import deepface
    print(f"✅ DeepFace version {deepface.__version__} is available.")
except ImportError:
    print("⚠️ DeepFace is NOT installed. To run DeepFace verification, run:")
    print("   !pip install deepface tf-keras")

✅ Project Root: C:\Users\priya\Desktop\prototype
✅ DeepFace version 0.0.101 is available.


## 1. Manual Pairwise Face Verification Setup
The cell below is pre-filled with the images in your `face_verification` folder (`passport2.jpg` & `current_image.jpg`).

In [2]:
# ==============================================================================
# MANUAL TESTING CONFIGURATION — PRE-FILLED WITH YOUR FACE VERIFICATION IMAGES
# ==============================================================================

PASSPORT_PHOTO_PATH = str(PROJECT_ROOT / "face_verification" / "passport2.jpg")
CURRENT_FACE_PHOTO_PATH = str(PROJECT_ROOT / "face_verification" / "current_image.jpg")

PASSPORT_NUMBER = None    # e.g., "A1234567" or None if OCR failed
OCR_ISSUE_DATE = None     # e.g., "2019-03-20" or None if OCR failed
REFERENCE_DATE = date.today()  # Override with date(2026, 9, 20) for reproducible testing

print(f"Passport Photo Path: {PASSPORT_PHOTO_PATH}")
print(f"Current Photo Path:  {CURRENT_FACE_PHOTO_PATH}")
print(f"Passport Number:     {PASSPORT_NUMBER}")
print(f"OCR Issue Date:      {OCR_ISSUE_DATE}")

Passport Photo Path: C:\Users\priya\Desktop\prototype\face_verification\passport2.jpg
Current Photo Path:  C:\Users\priya\Desktop\prototype\face_verification\current_image.jpg
Passport Number:     None
OCR Issue Date:      None


In [3]:
# Execute Module 4 Face Verification Pipeline on Provided Image Paths
report = verify_faces_with_context(
    passport_photo_path=PASSPORT_PHOTO_PATH,
    current_face_photo_path=CURRENT_FACE_PHOTO_PATH,
    passport_number=PASSPORT_NUMBER,
    ocr_issue_date=OCR_ISSUE_DATE,
    reference_date=REFERENCE_DATE,
)

print("\n================ MODULE 4 FACE VERIFICATION REPORT ================")
print(f"Pipeline ID:               {report.get('pipeline_id')}")
print(f"DB Lookup Status:          {report.get('database_lookup_status')}")
print(f"DB Record Found:           {report.get('database_record_found')} ({report.get('database_table_source')})")
print(f"Resolved Issue Date:       {report.get('resolved_issue_date')} (Source: {report.get('issue_date_source')})")
print(f"Document Elapsed Age:      {report.get('elapsed_years')} years ({report.get('elapsed_days')} days)")
print(f"Face Verification Status:  {report.get('face_verification_status')}")
print(f"Raw Distance:              {report.get('raw_distance')} (Threshold: {report.get('model_threshold')})")
print(f"Derived Similarity Score:  {report.get('derived_similarity_score')}%")
print(f"Faces Detected (Pass/Curr):{report.get('passport_face_count')} / {report.get('current_face_count')}")
print(f"Age Context Policy Case:   {report.get('policy_case')}")
print(f"Manual Review Required:    {report.get('manual_review_recommended')}")
print(f"\nExplanation:\n{report.get('human_readable_explanation')}")

26-09-21 00:48:08 - ⚠️ 
 ⚠️ DEPRECATION WARNING:
 Running 'pip install deepface' alone will no longer be sufficient and will
 NOT install a default backend in an upcoming major release.

 Currently, TensorFlow is included by default, but this behavior will be deprecated.
 Please explicitly specify your preferred backend engine when installing:

   -> pip install deepface[tensorflow]
   -> pip install deepface[pytorch]

 Otherwise, you will encounter 'module not found' errors.



================ MODULE 4 FACE VERIFICATION REPORT ================
Pipeline ID:               ef93e187-41bf-4036-9423-5d36719dd093
DB Lookup Status:          MISSING_PASSPORT_NUMBER
DB Record Found:           False (None)
Resolved Issue Date:       UNAVAILABLE (Source: UNAVAILABLE)
Document Elapsed Age:      None years (None days)
Face Verification Status:  MATCH
Raw Distance:              0.2089 (Threshold: 0.68)
Derived Similarity Score:  84.64%
Faces Detected (Pass/Curr):1 / 1
Age Context Policy Case:   MAT

--- 
## 2. Comprehensive Test Suite (12 Required Scenarios)

The test suite below executes controlled test cases for all 12 policy and lookup scenarios defined in the project specification. Run each cell sequentially to verify system logic.

In [4]:
# Test Helper Function to Pretty Print Test Case Results
def print_test_case_summary(test_num, title, report):
    print(f"\n{'='*70}")
    print(f"TEST CASE {test_num}: {title}")
    print(f"{'='*70}")
    print(f"  • DB Lookup Status:         {report.get('database_lookup_status')}")
    print(f"  • Resolved Issue Date:      {report.get('resolved_issue_date')} (Source: {report.get('issue_date_source')})")
    print(f"  • Document Elapsed Age:     {report.get('elapsed_years')} years")
    print(f"  • Verification Status:      {report.get('face_verification_status')}")
    print(f"  • Raw Distance / Threshold: {report.get('raw_distance')} / {report.get('model_threshold')}")
    print(f"  • Policy Case:              {report.get('policy_case')}")
    print(f"  • Manual Review Required:   {report.get('manual_review_recommended')}")
    print(f"  • Explanation:\n    {report.get('human_readable_explanation')}")

### Test Case 1: Passport number found in DB and issue date available

In [5]:
# Scenario 1: Passport found in SQLite (using synthetic seed record 'A1234567')
db_status, db_rec, src, err = lookup_passport_record("A1234567")
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=db_rec, ocr_issue_date=None, reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.35, "threshold": 0.68, "derived_similarity_score": 74.26}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc1 = {
    "database_lookup_status": db_status, "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(1, "Passport number found in DB and issue date available", report_tc1)


TEST CASE 1: Passport number found in DB and issue date available
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      UNAVAILABLE (Source: UNAVAILABLE)
  • Document Elapsed Age:     None years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.35 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 74.26%, distance: 0.35 <= threshold: 0.68).


### Test Case 2: Passport number not found in DB

In [6]:
# Scenario 2: Passport number extracted but not present in database
db_status, db_rec, src, err = lookup_passport_record("NONEXISTENT999")
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=db_rec, ocr_issue_date="2022-01-10", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.40, "threshold": 0.68, "derived_similarity_score": 70.59}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc2 = {
    "database_lookup_status": db_status, "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(2, "Passport number not found in DB", report_tc2)


TEST CASE 2: Passport number not found in DB
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2022-01-10 (Source: OCR)
  • Document Elapsed Age:     4.69 years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.4 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 70.59%, distance: 0.4 <= threshold: 0.68). Document age: 4.7 years.


### Test Case 3: Passport number unavailable because OCR failed

In [7]:
# Scenario 3: Passport number is None (OCR failed)
db_status, db_rec, src, err = lookup_passport_record(None)
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2020-05-15", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.38, "threshold": 0.68, "derived_similarity_score": 72.06}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc3 = {
    "database_lookup_status": db_status, "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(3, "Passport number unavailable because OCR failed", report_tc3)


TEST CASE 3: Passport number unavailable because OCR failed
  • DB Lookup Status:         MISSING_PASSPORT_NUMBER
  • Resolved Issue Date:      2020-05-15 (Source: OCR)
  • Document Elapsed Age:     6.35 years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.38 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 72.06%, distance: 0.38 <= threshold: 0.68). Document age: 6.3 years.


### Test Case 4: Database record found but issue date missing

In [8]:
# Scenario 4: DB record found but date_of_issue is missing/null
db_rec_nodate = {"passport_number": "TEST123", "name": "TEST USER", "date_of_issue": None}
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=db_rec_nodate, ocr_issue_date="2019-11-20", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.30, "threshold": 0.68, "derived_similarity_score": 77.94}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc4 = {
    "database_lookup_status": "FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(4, "Database record found but issue date missing", report_tc4)


TEST CASE 4: Database record found but issue date missing
  • DB Lookup Status:         FOUND
  • Resolved Issue Date:      2019-11-20 (Source: OCR)
  • Document Elapsed Age:     6.83 years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.3 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 77.94%, distance: 0.3 <= threshold: 0.68). Document age: 6.8 years.


### Test Case 5: Database record unavailable and OCR issue date unavailable

In [9]:
# Scenario 5: Neither DB nor OCR provides an issue date
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date=None, reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.32, "threshold": 0.68, "derived_similarity_score": 76.47}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc5 = {
    "database_lookup_status": "MISSING_PASSPORT_NUMBER", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(5, "Database record unavailable and OCR issue date unavailable", report_tc5)


TEST CASE 5: Database record unavailable and OCR issue date unavailable
  • DB Lookup Status:         MISSING_PASSPORT_NUMBER
  • Resolved Issue Date:      UNAVAILABLE (Source: UNAVAILABLE)
  • Document Elapsed Age:     None years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.32 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 76.47%, distance: 0.32 <= threshold: 0.68).


### Test Case 6: Low similarity with an old issue date (Case B — Possible Aging)

In [10]:
# Scenario 6: Distance 0.78 > 0.68 (MISMATCH) and document issued 11.5 years ago (>= 5 year threshold)
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2015-03-15", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MISMATCH", "is_match": False, "raw_distance": 0.78, "threshold": 0.68, "derived_similarity_score": 42.65}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src, config=AgePolicyConfig(aging_threshold_years=5.0))

report_tc6 = {
    "database_lookup_status": "NOT_FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(6, "Low similarity with an old issue date (Possible Aging)", report_tc6)


TEST CASE 6: Low similarity with an old issue date (Possible Aging)
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2015-03-15 (Source: OCR)
  • Document Elapsed Age:     11.52 years
  • Verification Status:      MISMATCH
  • Raw Distance / Threshold: 0.78 / 0.68
  • Policy Case:              MISMATCH_POSSIBLE_AGING
  • Manual Review Required:   True
  • Explanation:
    Face similarity is below the threshold (derived score: 42.65%, distance: 0.78 > threshold: 0.68). The passport was issued 11.5 years ago (>= 5.0 year threshold). Natural facial changes over time may be one possible factor contributing to the similarity difference. This cannot be confirmed automatically. Manual verification is required.


### Test Case 7: Low similarity with an unavailable issue date (Case D — Exact Prompt Wording)

In [11]:
# Scenario 7: Low similarity and issue date unavailable
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date=None, reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MISMATCH", "is_match": False, "raw_distance": 0.82, "threshold": 0.68, "derived_similarity_score": 39.71}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc7 = {
    "database_lookup_status": "MISSING_PASSPORT_NUMBER", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(7, "Low similarity with an unavailable issue date (Case D Wording)", report_tc7)


TEST CASE 7: Low similarity with an unavailable issue date (Case D Wording)
  • DB Lookup Status:         MISSING_PASSPORT_NUMBER
  • Resolved Issue Date:      UNAVAILABLE (Source: UNAVAILABLE)
  • Document Elapsed Age:     None years
  • Verification Status:      MISMATCH
  • Raw Distance / Threshold: 0.82 / 0.68
  • Policy Case:              MISMATCH_UNKNOWN_AGE
  • Manual Review Required:   True
  • Explanation:
    Face similarity is below the configured threshold. The passport issue date could not be verified, so the time elapsed since issuance is unknown. Age-related facial changes may be one possible factor contributing to the low similarity score, but this cannot be confirmed. Manual verification is required.


### Test Case 8: High similarity with an old issue date (Case A)

In [12]:
# Scenario 8: High similarity (distance 0.30 <= 0.68) on a 10-year old passport
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2016-01-10", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "MATCH", "is_match": True, "raw_distance": 0.30, "threshold": 0.68, "derived_similarity_score": 77.94}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc8 = {
    "database_lookup_status": "NOT_FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(8, "High similarity with an old issue date (No aging warning)", report_tc8)


TEST CASE 8: High similarity with an old issue date (No aging warning)
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2016-01-10 (Source: OCR)
  • Document Elapsed Age:     10.69 years
  • Verification Status:      MATCH
  • Raw Distance / Threshold: 0.3 / 0.68
  • Policy Case:              MATCH_HIGH_SIMILARITY
  • Manual Review Required:   False
  • Explanation:
    Face verification PASSED. High similarity confirmed (derived score: 77.94%, distance: 0.3 <= threshold: 0.68). Document age: 10.7 years.


### Test Case 9: Borderline similarity (Case E)

In [13]:
# Scenario 9: Distance 0.70 is slightly above threshold 0.68 (within borderline margin 0.05)
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2021-06-15", reference_date=date(2026, 9, 20))

fake_verif = {"verification_status": "BORDERLINE", "is_match": False, "raw_distance": 0.70, "threshold": 0.68, "derived_similarity_score": 48.53}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc9 = {
    "database_lookup_status": "NOT_FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(9, "Borderline similarity", report_tc9)


TEST CASE 9: Borderline similarity
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2021-06-15 (Source: OCR)
  • Document Elapsed Age:     5.26 years
  • Verification Status:      BORDERLINE
  • Raw Distance / Threshold: 0.7 / 0.68
  • Policy Case:              BORDERLINE_SIMILARITY
  • Manual Review Required:   True
  • Explanation:
    Face verification is BORDERLINE (derived score: 48.53%, distance: 0.7, threshold: 0.68). Verification outcome is inconclusive. Manual review by an authorized officer is required.


### Test Case 10: No face detected (Case F)

In [14]:
# Scenario 10: 0 faces detected in image
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2020-01-01", reference_date=date(2026, 9, 20))

fake_verif = {
    "verification_status": "NO_FACE_DETECTED", "is_match": None, "raw_distance": None, "threshold": 0.68,
    "derived_similarity_score": None, "passport_face_count": 0, "current_face_count": 1,
    "error_message": "No face detected in Passport Photo (Image A)"
}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc10 = {
    "database_lookup_status": "NOT_FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(10, "No face detected", report_tc10)


TEST CASE 10: No face detected
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2020-01-01 (Source: OCR)
  • Document Elapsed Age:     6.72 years
  • Verification Status:      NO_FACE_DETECTED
  • Raw Distance / Threshold: None / 0.68
  • Policy Case:              VERIFICATION_UNAVAILABLE
  • Manual Review Required:   True
  • Explanation:
    Face verification could not be completed (NO_FACE_DETECTED). Reason: No face detected in Passport Photo (Image A) Manual verification required.


### Test Case 11: Multiple faces detected (Case F)

In [15]:
# Scenario 11: Multiple faces (2 faces) detected in image
parsed_date, date_str, date_src, el_y, el_d, warn = resolve_issue_date(db_record=None, ocr_issue_date="2020-01-01", reference_date=date(2026, 9, 20))

fake_verif = {
    "verification_status": "MULTIPLE_FACES_DETECTED", "is_match": None, "raw_distance": None, "threshold": 0.68,
    "derived_similarity_score": None, "passport_face_count": 1, "current_face_count": 2,
    "error_message": "Face verification aborted: Exactly 1 face is required per image. | Multiple faces (2) detected in Current Face Photo (Image B)"
}
policy_res = apply_age_context_policy(fake_verif, parsed_date, el_y, date_src)

report_tc11 = {
    "database_lookup_status": "NOT_FOUND", "resolved_issue_date": date_str, "issue_date_source": date_src,
    "elapsed_years": el_y, "face_verification_status": fake_verif["verification_status"],
    "raw_distance": fake_verif["raw_distance"], "model_threshold": fake_verif["threshold"],
    "policy_case": policy_res["policy_case"], "manual_review_recommended": policy_res["manual_review_recommended"],
    "human_readable_explanation": policy_res["explanation"],
}
print_test_case_summary(11, "Multiple faces detected", report_tc11)


TEST CASE 11: Multiple faces detected
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      2020-01-01 (Source: OCR)
  • Document Elapsed Age:     6.72 years
  • Verification Status:      MULTIPLE_FACES_DETECTED
  • Raw Distance / Threshold: None / 0.68
  • Policy Case:              VERIFICATION_UNAVAILABLE
  • Manual Review Required:   True
  • Explanation:
    Face verification could not be completed (MULTIPLE_FACES_DETECTED). Reason: Face verification aborted: Exactly 1 face is required per image. | Multiple faces (2) detected in Current Face Photo (Image B) Manual verification required.


### Test Case 12: Invalid or unreadable image path (Case F)

In [16]:
# Scenario 12: Image path is invalid or file does not exist
report_tc12 = verify_faces_with_context(
    passport_photo_path="invalid_path_to_non_existent_file.jpg",
    current_face_photo_path="invalid_path_to_non_existent_file2.jpg",
    passport_number="A1234567",
    reference_date=date(2026, 9, 20),
)
print_test_case_summary(12, "Invalid or unreadable image path", report_tc12)

Image validation failed: Passport photo (Image A): File not found at path 'invalid_path_to_non_existent_file.jpg'. | Current face photo (Image B): File not found at path 'invalid_path_to_non_existent_file2.jpg'.



TEST CASE 12: Invalid or unreadable image path
  • DB Lookup Status:         NOT_FOUND
  • Resolved Issue Date:      UNAVAILABLE (Source: UNAVAILABLE)
  • Document Elapsed Age:     None years
  • Verification Status:      INVALID_IMAGE
  • Raw Distance / Threshold: None / None
  • Policy Case:              VERIFICATION_UNAVAILABLE
  • Manual Review Required:   True
  • Explanation:
    Face verification could not be completed (INVALID_IMAGE). Reason: Passport photo (Image A): File not found at path 'invalid_path_to_non_existent_file.jpg'. | Current face photo (Image B): File not found at path 'invalid_path_to_non_existent_file2.jpg'. Manual verification required.
